# Análisis de un grupo de Telegram

En esta libreta analizo el historial de un grupo público de Telegram en ruso, dedicado a
la compraventa e intercambio de cuentas de un videojuego, entre agosto de 2025 y
septiembre de 2026. Los datos crudos no se suben al repositorio, así que la libreta se
entrega ya ejecutada. Para proteger a los participantes, los nombres se reemplazaron por
superhéroes y solo se muestran resultados agregados, nunca mensajes completos.

## 0. Configuración

Cargo las librerías y las rutas del proyecto, que están definidas en `src/config.py`.

In [ ]:
import re
import sys
from pathlib import Path

import emoji
import pandas as pd
from lxml import html

# Para poder importar src/ desde la carpeta notebooks/
PROJ_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJ_ROOT))

from src.config import CHAT_EXPORT_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR

pd.set_option("display.max_columns", 30)

## 1. Lectura del HTML exportado

Telegram exportó el chat en 35 archivos HTML (`messages.html`, `messages2.html`, ...,
`messages35.html`). Los leo en orden numérico para conservar la secuencia de los mensajes.

In [ ]:
def numero_archivo(ruta):
    # messages.html es el 1, messages2.html el 2, etc.
    return int(re.search(r"messages(\d*)\.html", ruta.name).group(1) or 1)


archivos = sorted(CHAT_EXPORT_DIR.glob("messages*.html"), key=numero_archivo)
print(f"{len(archivos)} archivos: {archivos[0].name} ... {archivos[-1].name}")

Cada mensaje de usuario es un bloque `div` con las clases `message default`; los avisos
del sistema, como los separadores de fecha, se ignoran. La multimedia no se descargó, así
que de cada foto, video o sticker solo conozco su tipo y, en los stickers, el emoji
asociado. En audios y archivos adjuntos el título es el nombre de la canción o del
archivo, por eso el tipo se deduce de la clase del bloque y ese título no se guarda.
Cuando alguien menciona a otra persona que no tiene `@usuario`, Telegram escribe su nombre
real en el texto; esas menciones se reemplazan desde aquí por `@usuario_anon`.

In [ ]:
# Tipos que Telegram escribe como título del bloque multimedia
TITULOS_MEDIA = {"Photo", "Sticker", "Video file", "Animation", "Voice message", "Video message"}


def hijo(elemento, clases):
    # Primer hijo directo que tiene todas las clases indicadas
    buscadas = set(clases.split())
    for h in elemento:
        if buscadas <= set((h.get("class") or "").split()):
            return h
    return None


def extraer_texto(div_texto):
    # Los saltos de línea vienen como <br>
    for br in div_texto.iter("br"):
        br.tail = "\n" + (br.tail or "")
    # Menciones sin @usuario: el enlace muestra el nombre real de la persona
    for a in div_texto.xpath('.//a[contains(@onclick, "ShowMentionName")]'):
        a.text = "@usuario_anon"
    return div_texto.text_content().strip()


def tipo_de_media(media):
    if media.find_class("media_poll"):
        return "Poll"
    titulo = media.find_class("title")[0].text_content().strip()
    if titulo in TITULOS_MEDIA:
        return titulo
    if media.find_class("media_audio_file"):
        return "Audio file"
    if media.find_class("media_file"):
        return "File"
    return "Other"


def contar_reacciones(body):
    total = 0
    for reaccion in body.find_class("reaction"):
        conteo = reaccion.find_class("count")
        if conteo:
            total += int(conteo[0].text_content())
        else:
            # Con pocas reacciones se muestran avatares en vez de un número
            total += len(reaccion.find_class("userpic"))
    return total


def leer_mensaje(msg):
    body = hijo(msg, "body")
    # En los reenviados, el texto y la multimedia están dentro de otro bloque
    reenviado = hijo(body, "forwarded body")
    contenido = reenviado if reenviado is not None else body

    nombre = hijo(body, "from_name")
    texto = hijo(contenido, "text")
    media = hijo(contenido, "media_wrap")
    respuesta = hijo(body, "reply_to")

    tipo_media = tipo_de_media(media) if media is not None else None
    sticker_emoji = None
    if tipo_media == "Sticker":
        # El estado del sticker se ve así: "<emoji>, <tamaño>"
        sticker_emoji = media.find_class("status")[0].text_content().split(",")[0].strip()
    responde_a_id = None
    if respuesta is not None:
        # El enlace es "#go_to_message33351" o "messages2.html#go_to_message33351"
        href = respuesta.find(".//a").get("href")
        responde_a_id = int(href.split("go_to_message")[1])

    return {
        "id_mensaje": int(msg.get("id").replace("message", "")),
        "fecha": hijo(body, "date").get("title"),
        "nombre": nombre.text_content().strip() if nombre is not None else None,
        "es_continuacion": "joined" in msg.get("class").split(),
        "reenviado": reenviado is not None,
        "es_respuesta": respuesta is not None,
        "responde_a_id": responde_a_id,
        "texto": extraer_texto(texto) if texto is not None else None,
        "tipo_media": tipo_media,
        "sticker_emoji": sticker_emoji,
        "n_reacciones": contar_reacciones(body),
    }

Los mensajes seguidos de una misma persona no repiten el nombre, así que lo relleno con
el último remitente visto. Algunos nombres aparecen como "Nombre via @bot"; me quedo con
el nombre y marco esos mensajes en la columna `via_bot`. Las horas vienen en UTC-07:00; esto se revisa en la sección de
días y horas.

In [ ]:
filas = []
for ruta in archivos:
    raiz = html.parse(str(ruta)).getroot()
    for msg in raiz.find_class("message"):
        if "default" in msg.get("class").split():
            fila = leer_mensaje(msg)
            fila["archivo"] = numero_archivo(ruta)
            filas.append(fila)

df_crudo = pd.DataFrame(filas)
df_crudo["timestamp"] = pd.to_datetime(df_crudo["fecha"], format="%d.%m.%Y %H:%M:%S UTC%z")
df_crudo["remitente"] = df_crudo["nombre"].ffill()
df_crudo["via_bot"] = df_crudo["remitente"].str.contains(" via @", regex=False)
df_crudo["remitente"] = df_crudo["remitente"].str.replace(r" via @\w+$", "", regex=True)
df_crudo["responde_a_id"] = df_crudo["responde_a_id"].astype("Int64")

df_crudo = df_crudo[[
    "archivo", "id_mensaje", "timestamp", "remitente", "via_bot", "es_continuacion",
    "reenviado", "es_respuesta", "responde_a_id", "texto", "tipo_media",
    "sticker_emoji", "n_reacciones",
]]
print(f"{len(df_crudo):,} mensajes leídos")

### Verificación de la lectura

La columna `remitente` todavía tiene los nombres reales, así que no la muestro; se
anonimiza en la sección 2. Por ahora reviso solo conteos generales.

In [ ]:
resumen = pd.Series({
    "Mensajes": len(df_crudo),
    "Remitentes distintos": df_crudo["remitente"].nunique(),
    "Mensajes de 'Deleted Account'": (df_crudo["remitente"] == "Deleted Account").sum(),
    "Con texto": df_crudo["texto"].notna().sum(),
    "Sin texto": df_crudo["texto"].isna().sum(),
    "Con multimedia": df_crudo["tipo_media"].notna().sum(),
    "Reenviados": df_crudo["reenviado"].sum(),
    "Respuestas": df_crudo["es_respuesta"].sum(),
    "Enviados con bot": df_crudo["via_bot"].sum(),
    "Con reacciones": (df_crudo["n_reacciones"] > 0).sum(),
})
print("Del", df_crudo["timestamp"].min(), "al", df_crudo["timestamp"].max())
resumen.to_frame("conteo")

In [ ]:
df_crudo["tipo_media"].value_counts(dropna=False).to_frame("mensajes")

In [ ]:
# Vista previa sin nombres ni textos
df_crudo.drop(columns=["remitente", "texto"]).head()

Se leyeron 34,868 mensajes de 1,322 remitentes. "Deleted Account" junta a todas las
cuentas eliminadas (3,073 mensajes), así que después la excluiré de los rankings por
usuario. Casi un 38 % de los mensajes no tiene texto y la gran mayoría de la multimedia
son fotos, lo que cuadra con un grupo donde se muestran capturas de cuentas en venta.

### Guardado intermedio

Guardo la tabla en `data/interim/` para no tener que volver a leer el HTML. Esa carpeta no se sube a GitHub porque todavía tiene los nombres reales.

In [ ]:
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
df_crudo.to_pickle(INTERIM_DATA_DIR / "mensajes_crudos.pkl")

## 2. Anonimización y formato tidy

Antes de cualquier análisis quito todo lo que pueda identificar a una persona: nombres,
menciones, correos, enlaces, tarjetas y teléfonos. Después armo la tabla tidy con la que se responden todas las preguntas.

### Nombres de superhéroes

Cada remitente recibe un nombre de superhéroe según su actividad: el más activo es Batman,
el segundo Superman, y así sucesivamente. Como hay más de 1,300 usuarios y solo 50 nombres,
a partir del usuario 51 se repiten con un sufijo (`Batman_2`, `Superman_2`, ...). "Deleted
Account" no es una persona sino todas las cuentas eliminadas juntas, así que la llamo
"Cuenta eliminada" y no le asigno superhéroe.

In [ ]:
HEROES = [
    "Batman", "Superman", "Flash", "Hulk", "Storm", "Thor", "Robin", "Spiderman",
    "Wolverine", "Aquaman", "Cyclops", "Rogue", "Falcon", "Groot", "Vision",
    "Hawkeye", "Blade", "Raven", "Cyborg", "Gambit", "Nightwing", "Batgirl",
    "Supergirl", "Daredevil", "Elektra", "Punisher", "Shazam", "Atom", "Starfire",
    "Beast", "Iceman", "Colossus", "Nova", "Quicksilver", "Wasp", "Hercules",
    "Magik", "Havok", "Bishop", "Cable", "Domino", "Deadpool", "Sentry",
    "Hellboy", "Zatanna", "Huntress", "Stargirl", "Mera", "Firestorm", "Rocket",
]

# Remitentes ordenados por número de mensajes (en empate, el que escribió primero)
actividad = (
    df_crudo[df_crudo["remitente"] != "Deleted Account"]
    .groupby("remitente")
    .agg(mensajes=("id_mensaje", "size"), primer_mensaje=("id_mensaje", "min"))
    .sort_values(["mensajes", "primer_mensaje"], ascending=[False, True])
)

# La tabla de equivalencias solo existe en memoria; no se guarda en ningún archivo
equivalencias = {}
for i, remitente in enumerate(actividad.index):
    vuelta = i // len(HEROES) + 1
    heroe = HEROES[i % len(HEROES)]
    equivalencias[remitente] = heroe if vuelta == 1 else f"{heroe}_{vuelta}"
equivalencias["Deleted Account"] = "Cuenta eliminada"

print(f"{len(actividad):,} usuarios anonimizados; el sufijo más alto es _{vuelta}")
print("Los 15 más activos:", ", ".join(list(equivalencias.values())[:15]))

### Limpieza del texto

En el texto cambio cada mención `@usuario` por `@usuario_anon` y borro correos, enlaces
(incluidos los de `t.me`, que suelen llevar al perfil de alguien), números de tarjeta y
teléfonos. Como tarjeta cuento 4 grupos de 4 dígitos separados por espacios o guiones, y
como teléfono el formato ruso (`+7` u `8` seguido de 10 dígitos), los números que empiezan
con `+` y cualquier número de 10 dígitos o más. Los números cortos, como precios y
cantidades, se conservan.

In [ ]:
PATRON_CORREO = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
PATRON_ENLACE = (
    r"(?:https?://|www\.)\S+"
    r"|\b(?:t|telegram)\.me/\S+"
    r"|\b[\w-]+\.(?:ru|com|net|org|me|su|gg|io)\b(?:/\S*)?"
)
PATRON_MENCION = r"@\w+"
PATRON_TARJETA = r"\b\d{4}(?:[ -]\d{4}){3}\b"
PATRON_TELEFONO = (
    r"(?:\+7|\b8)[\s(-]*\d{3}[\s)-]*\d{3}[\s-]*\d{2}[\s-]*\d{2}\b"
    r"|\+\d[\d\s()-]{8,}\d"
    r"|\b\d{10,}\b"
)


def limpiar_texto(texto):
    # El correo va antes que los enlaces y las menciones para no dejar la parte del usuario
    texto = texto.str.replace(PATRON_CORREO, " ", regex=True)
    texto = texto.str.replace(PATRON_ENLACE, " ", regex=True, flags=re.IGNORECASE)
    texto = texto.str.replace(PATRON_MENCION, "@usuario_anon", regex=True)
    texto = texto.str.replace(PATRON_TARJETA, " ", regex=True)
    texto = texto.str.replace(PATRON_TELEFONO, " ", regex=True)
    texto = texto.str.replace(r"[ \t]+", " ", regex=True).str.strip()
    # Si el mensaje solo tenía un enlace, queda vacío y lo trato como sin texto
    return texto.mask(texto == "")


texto_limpio = limpiar_texto(df_crudo["texto"])

# Comprobación: no debe quedar ningún dato de contacto ni mención con nombre
revision = pd.Series({
    "Correos": texto_limpio.str.contains(PATRON_CORREO).sum(),
    "Enlaces": texto_limpio.str.contains(PATRON_ENLACE, flags=re.IGNORECASE).sum(),
    "Tarjetas": texto_limpio.str.contains(PATRON_TARJETA).sum(),
    "Teléfonos": texto_limpio.str.contains(PATRON_TELEFONO).sum(),
    "Menciones sin anonimizar": texto_limpio.str.contains(r"@(?!usuario_anon\b)\w+").sum(),
    "Mensajes con menciones anonimizadas": texto_limpio.str.contains("@usuario_anon", regex=False).sum(),
    "Mensajes que quedaron sin texto": (df_crudo["texto"].notna() & texto_limpio.isna()).sum(),
})
revision.to_frame("mensajes")

### Construcción de la tabla tidy

Además del texto limpio, calculo algunas columnas que se usan en las preguntas. El tipo de
contenido se toma de la multimedia; si no hay multimedia pero sí texto, el mensaje es de
texto. Cuento como palabra cualquier secuencia de letras latinas o cirílicas, sin contar números, emojis ni menciones. Por último, marco como
repetido todo texto que aparece idéntico más de 3 veces en el grupo, que casi siempre son
anuncios pegados una y otra vez.

In [ ]:
TIPOS = {
    "Photo": "foto", "Sticker": "sticker", "Animation": "gif",
    "Video file": "video", "Video message": "video", "Voice message": "voz",
}
PATRON_PALABRA = r"[a-zа-яё]+(?:-[a-zа-яё]+)*"


def tipo_contenido(tipo_media, texto):
    if pd.notna(tipo_media):
        return TIPOS.get(tipo_media, "otro")  # audios, archivos y encuestas son "otro"
    if pd.notna(texto):
        return "texto"
    return "otro"  # mensajes que solo tenían un enlace


df = pd.DataFrame({
    "id_mensaje": df_crudo["id_mensaje"],
    "timestamp": df_crudo["timestamp"],
    "usuario": df_crudo["remitente"].map(equivalencias),
    "texto": texto_limpio,
    "tipo_contenido": [tipo_contenido(m, t) for m, t in zip(df_crudo["tipo_media"], texto_limpio)],
    "reenviado": df_crudo["reenviado"],
    "es_respuesta": df_crudo["es_respuesta"],
    "responde_a_id": df_crudo["responde_a_id"],
    "sticker_emoji": df_crudo["sticker_emoji"],
    "n_reacciones": df_crudo["n_reacciones"],
    "via_bot": df_crudo["via_bot"],
})

veces_texto = df["texto"].map(df["texto"].value_counts())
df["es_repetido"] = veces_texto > 3

sin_menciones = df["texto"].str.replace("@usuario_anon", " ", regex=False)
df["n_palabras"] = sin_menciones.str.count(PATRON_PALABRA, flags=re.IGNORECASE).fillna(0).astype(int)
df["n_emojis"] = df["texto"].fillna("").map(emoji.emoji_count)

print(f"{len(df):,} filas y {df.shape[1]} columnas; usuarios sin nombre: {df['usuario'].isna().sum()}")
# Vista previa sin texto ni usuario: junto con la hora exacta podrían identificar a alguien
df.drop(columns=["texto", "usuario"]).head()

In [ ]:
tipos = df["tipo_contenido"].value_counts().to_frame("mensajes")
tipos["porcentaje"] = (100 * tipos["mensajes"] / len(df)).round(1)
tipos

In [ ]:
pd.Series({
    "Mensajes con texto": df["texto"].notna().sum(),
    "Mensajes repetidos (texto idéntico > 3 veces)": df["es_repetido"].sum(),
    "Textos distintos que se repiten": df.loc[df["es_repetido"], "texto"].nunique(),
    "Palabras en total": df["n_palabras"].sum(),
    "Palabras en mensajes no repetidos": df.loc[~df["es_repetido"], "n_palabras"].sum(),
    "Emojis en total": df["n_emojis"].sum(),
    "Mensajes de 'Cuenta eliminada'": (df["usuario"] == "Cuenta eliminada").sum(),
    "Fotos con descripción": ((df["tipo_contenido"] == "foto") & df["texto"].notna()).sum(),
}).to_frame("conteo")

El 80 % de los mensajes son fotos (muchas con un texto como descripción) y solo el 17.5 %
son mensajes únicamente de texto; stickers, GIFs y audios casi no se usan. De los 21,730
mensajes con texto, 8,404 son repeticiones de apenas 592 textos distintos, y quitarlos deja
160 mil de las 256 mil palabras. Por eso en el análisis de palabras se excluyen los
repetidos, porque si no los anuncios dominarían los resultados.

### ¿Por qué la tabla es tidy?

La tabla cumple las tres reglas de los datos tidy: cada fila es un mensaje, cada columna es
una sola variable y cada celda guarda un único valor. Por ejemplo, el tipo de contenido
está en una sola columna en lugar de tener una columna por tipo, y la fecha y la hora
están juntas en `timestamp`, de donde se pueden sacar el día o la hora cuando se necesiten.
La descripción de cada columna está en el
[diccionario de datos](../references/diccionario_de_datos.md).

El flujo completo es: los HTML de `data/raw/` se leen a una tabla cruda que se guarda en
`data/interim/`; después se anonimiza y limpia, y la tabla tidy resultante se guarda en
`data/processed/`, de donde parte todo el análisis.

### Guardado de la tabla limpia

La tabla tidy se guarda en `data/processed/`. Aunque ya no tiene nombres reales, sí tiene
los textos de los mensajes, así que esa carpeta tampoco se sube a GitHub. A partir de aquí
todo el análisis usa solo `df`.

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
df.to_pickle(PROCESSED_DATA_DIR / "mensajes_tidy.pkl")
df.dtypes.to_frame("tipo")